In [1]:
import os
import pandas as pd
import sqlite3

In [2]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DB_PATH = os.path.join(BASE_DIR, "2_DATABASE", "eko_cards.db")
PRICES_PATH = os.path.join(BASE_DIR, "3_PROJECT_DATA", "prices.xlsx")

In [3]:
def transaction_import():

    # -------------------------
    # READ EXCEL
    # -------------------------
    data = pd.read_excel(
        os.path.join(BASE_DIR, "3_PROJECT_DATA", "eko_transactions.xlsx"),
        dtype={"Number": str}
    )

    # -------------------------
    # CLEAN COLUMN NAMES
    # -------------------------
    data.columns = data.columns.str.strip()

    # -------------------------
    # CLEAN MATERIAL
    # -------------------------
    data["Material"] = (
        data["Material"]
        .astype(str)
        .str.replace(r'^\d+\s*', '', regex=True)   # remove product code
        .str.replace(r"\s+", " ", regex=True)      # normalize spaces
        .str.strip()
        .str.upper()                               # normalize text
    )

    # -------------------------
    # CLEAN VEHICLE
    # -------------------------
    data["Name"] = (
        data["Name"]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )

    # -------------------------
    # SELECT COLUMNS
    # -------------------------
    data = data[
        [
            "Plant",
            "Name",
            "Number",
            "Material",
            "Date",
            "Bill.qty",
            "Bill.qty.1",
            "FinPr",
            "Tot Amount",
            "Auth.time",
            "Km stand"
        ]
    ]

    # -------------------------
    # FORMAT DATE
    # -------------------------
    data["Date"] = pd.to_datetime(
        data["Date"],
        dayfirst=True,
        errors="coerce"
    )

    data = data[data["Date"].notna()]
    data["Date"] = data["Date"].dt.strftime("%Y-%m-%d")

    # -------------------------
    # CLEAN CARD NUMBERS
    # -------------------------
    data["Number"] = (
        data["Number"]
        .astype(str)
        .str.replace(r"\D", "", regex=True)
    )

    data = data[data["Number"].str.len() > 0]

    # -------------------------
    # REMOVE NON TRANSACTIONS
    # -------------------------
    data["Bill.qty"] = pd.to_numeric(data["Bill.qty"], errors="coerce")
    data = data[data["Bill.qty"] > 0]

    # -------------------------
    # RENAME COLUMNS
    # -------------------------
    data = data.rename(columns={
        "Plant": "plant",
        "Name": "name",
        "Number": "card_number",
        "Material": "material",
        "Date": "date",
        "Bill.qty": "bill_qty",
        "Bill.qty.1": "bill_qty2",
        "FinPr": "price",
        "Tot Amount": "amount",
        "Auth.time": "auth_time",
        "Km stand": "Km_stand"
    })
    data.columns = data.columns.str.strip()
    # -------------------------
    # CONNECT 2_DATABASE
    # -------------------------
    conn = sqlite3.connect(DB_PATH)

    data.to_sql(
        "transactions",
        conn,
        if_exists="append",
        index=False
    )

    conn.close()

    print(f"Transactions imported successfully: {len(data)} rows")

In [4]:
def price_import():
    import sqlite3
    import pandas as pd

    # -------------------------
    # READ
    # -------------------------
    prices = pd.read_excel(PRICES_PATH)
    prices.columns = prices.columns.str.strip()

    prices = prices[["ДАТА", "ФИРМА", "ЕИК", "ПРОДУКТ", "КРАЙНА_ЦЕНА"]]

    prices = prices.rename(columns={
        "ДАТА": "date",
        "ФИРМА": "company",
        "ЕИК": "eik",
        "ПРОДУКТ": "product",
        "КРАЙНА_ЦЕНА": "final_price"
    })

    # -------------------------
    # CLEAN
    # -------------------------
    prices["company"] = (
        prices["company"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.upper()
    )

    prices["product"] = (
        prices["product"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.upper()
    )

    prices["eik"] = prices["eik"].astype(str).str.strip()

    prices["date"] = pd.to_datetime(prices["date"], dayfirst=True, errors="coerce")
    prices = prices[prices["date"].notna()]
    prices["date"] = prices["date"].dt.strftime("%Y-%m-%d")

    prices["final_price"] = pd.to_numeric(prices["final_price"], errors="coerce")
    prices = prices[prices["final_price"].notna()]

    # -------------------------
    # DB
    # -------------------------
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print("DB PATH:", DB_PATH)

    # check table
    table_info = pd.read_sql("PRAGMA table_info(prices)", conn)
    print("TABLE STRUCTURE BEFORE:")
    print(table_info)

    # add column if missing
    if "eik" not in table_info["name"].values:
        print("Adding eik column...")
        cursor.execute("ALTER TABLE prices ADD COLUMN eik TEXT")
        conn.commit()

    # verify again
    table_info = pd.read_sql("PRAGMA table_info(prices)", conn)
    print("TABLE STRUCTURE AFTER:")
    print(table_info)

    # -------------------------
    # INSERT (SAFE)
    # -------------------------
    insert_query = """
    INSERT INTO prices (date, company, product, final_price, eik)
    VALUES (?, ?, ?, ?, ?)
    """

    data = prices[["date", "company", "product", "final_price", "eik"]].values.tolist()

    print("SAMPLE DATA:")
    print(data[:5])

    cursor.executemany(insert_query, data)

    conn.commit()

    # -------------------------
    # VERIFY INSERT
    # -------------------------
    check = pd.read_sql("SELECT * FROM prices ORDER BY ROWID DESC LIMIT 5", conn)
    print("LAST INSERTED ROWS:")
    print(check)

    conn.close()

    print(f"Prices imported successfully: {len(data)} rows")

In [5]:
def company_import():

    cards = pd.read_excel("../3_PROJECT_DATA/cards.xlsx", dtype={"Number": str})

    cards = cards.loc[:, ~cards.columns.str.contains('^Unnamed')]

    cards.columns = cards.columns.str.strip()

    cards = cards.rename(columns={
        "Company": "company",
        "Name": "vehicle",
        "Number": "card_number"
    })


    cards["company"] = cards["company"].astype(str).str.strip()

    conn = sqlite3.connect(DB_PATH)

    cards.to_sql(
        "cards",
        conn,
        if_exists="replace",
        index=False
    )

    conn.close()

In [6]:
def merge_tables():
    import sqlite3
    import pandas as pd

    conn = sqlite3.connect(DB_PATH)

    query = """
    SELECT
        t.*,
        c.company,

        CAST(p.final_price * 100 + 0.5 AS INTEGER) / 100.0 AS final_price

    FROM transactions t

    LEFT JOIN cards c
        ON t.card_number = c.card_number

    LEFT JOIN prices p
        ON c.eik = p.eik
        AND UPPER(p.product) = UPPER(t.material)

        AND p.date = (
            SELECT MAX(p2.date)
            FROM prices p2
            WHERE p2.eik = c.eik
              AND UPPER(p2.product) = UPPER(t.material)
              AND p2.date <= t.date
        )
    """

    data = pd.read_sql(query, conn)
    conn.close()

    return data

In [7]:
def generate_result(data):

    os.makedirs(os.path.join(BASE_DIR, "A_FINAL_RESULT"), exist_ok=True)

    output_file = os.path.join(BASE_DIR, "A_FINAL_RESULT", "eko_transactions.xlsx")

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        for company, df in data.groupby("company", dropna=False):

            company_name = "Unknown" if pd.isna(company) else str(company)
            sheet_name = company_name[:31]

            df["price"] = pd.to_numeric(df["price"], errors="coerce").round(2)
            df["final_price"] = pd.to_numeric(df["final_price"], errors="coerce").round(2)

            df["eko_total"] = (df["bill_qty"] * df["price"]).round(2)
            df["gta_total"] = (df["bill_qty"] * df["final_price"]).round(2)

            df = df[
                [
                    "date",
                    "plant",
                    "name",
                    "card_number",
                    "material",
                    "bill_qty",
                    "bill_qty2",
                    "final_price",
                    "gta_total",
                    "price",
                    "eko_total",
                    "auth_time",
                    "Km_stand"
                ]
            ]

            df = df.rename(columns={
                "date": "Дата",
                "plant": "Станция",
                "name": "Име",
                "card_number": "Номер на карта",
                "material": "Име на артикул",
                "bill_qty": "Литри",
                "bill_qty2": "Тип количество",
                "final_price": "GTA цена",
                "gta_total": "Сума по GTA цена",
                "price": "ЕКО цена",
                "eko_total": "Сума по ЕКО цена",
                "auth_time": "Час",
                "Km_stand": "Километри"
            })

            df.to_excel(writer, sheet_name=sheet_name, index=False)

    print("Output file generated:", output_file)

In [8]:
def delete_transactions():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("DELETE FROM transactions")

    conn.commit()
    conn.close()

In [9]:
def delete_company():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("DELETE FROM cards")

    conn.commit()
    conn.close()

In [10]:
def delete_prices():

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("DELETE FROM prices")

    conn.commit()
    conn.close()

    print("All prices deleted.")

In [11]:
def delete_prices_by_month(year, month):

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    month_str = f"{year}-{str(month).zfill(2)}"

    cursor.execute(
        """
        DELETE FROM prices
        WHERE strftime('%Y-%m', date) = ?
        """,
        (month_str,)
    )

    conn.commit()
    conn.close()

    print(f"Prices deleted for: {month_str}")

In [12]:
def execute_pipeline():
    delete_transactions()

    transaction_import()

    data = merge_tables()

    generate_result(data)